In [26]:
import pandas as pd
import requests
from geopy.distance import geodesic

In [27]:
df = pd.read_csv("carreras.csv")
df

,Nombre de la carrera,Dirección,Área del conocimiento,Datos Estadísticos (DGAE-UNAM),Aciertos Mínimos por Examen de Selección,Promedio de corte pase reglamentado,Cupo,Características del aspirante,Descripción de la carrera,Palabras asociadas,Latitud,Longitud,Unnamed: 12
0,Actuaría - Facultad de Ciencias,"Facultad de Ciencias, Ciudad de México",Ciencias Físico - Matemáticas y de las Ingenie...,La demanda a primer ingreso en esta licenciatu...,Facultad de Ciencias: 109,Facultad de Ciencias: 9.00,Asignación total de 695 lugares (conjunto con ...,Haber cursado el Área de las Ciencias Físico-M...,"Los actuarios son profesionistas que estudian,...","actuaría, seguros, finanzas, demografía, estad...","19,3243","-99,1796",NaN
1,Actuaría - FES Acatlán,"FES Acatlán, Estado de México",Ciencias Físico - Matemáticas y de las Ingenie...,La demanda a primer ingreso en esta licenciatu...,FES Acatlán: 97,FES Acatlán: 8.73,Asignación total de 695 lugares (conjunto con ...,Haber cursado el Área de las Ciencias Físico-M...,"Los actuarios son profesionistas que estudian,...","actuaría, seguros, finanzas, demografía, estad...","19,5135","-99,2484",NaN
2,Administración - FCA Escolarizada,"Facultad de Contaduría y Administración, Ciuda...",Ciencias Sociales,La demanda a primer ingreso en esta licenciatu...,FCA Escolarizada: 90,FCA Escolarizada: 8.11,"Asignación total de 2,539 lugares (conjunto to...","Bases matemáticas, lectura y redacción, cienci...",En Administración se forman profesionistas con...,"administración, gestión, empresas, organizacio...","19,3285","-99,1882",NaN
3,Administración - FCA Abierta,Facultad de Contaduría y Administración (Siste...,Ciencias Sociales,La demanda a primer ingreso en esta licenciatu...,FCA Abierta: 78,FCA Abierta: 7.24,"Asignación total de 2,539 lugares (conjunto to...","Bases matemáticas, lectura y redacción, cienci...",En Administración se forman profesionistas con...,"administración, gestión, empresas, organizacio...","19,3285","-99,1882",NaN
4,Administración - FCA a Distancia,Facultad de Contaduría y Administración (Modal...,Ciencias Sociales,La demanda a primer ingreso en esta licenciatu...,FCA a Distancia: 75,FCA a Distancia: 8.27,"Asignación total de 2,539 lugares (conjunto to...","Bases matemáticas, lectura y redacción, cienci...",En Administración se forman profesionistas con...,"administración, gestión, empresas, organizacio...","19,3285","-99,1882",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,Tecnologías para la Información en Ciencias - ...,"ENES Unidad Morelia, Michoacán",Ciencias Físico - Matemáticas y de las Ingenie...,La demanda a primer ingreso en esta licenciatu...,ENES Unidad Morelia: 60,ENES Unidad Morelia: 7.50,Información no disponible,Haber cursado el Área de las Ciencias Físico-M...,El licenciado en Tecnologías para la Informaci...,"tecnologías de la información, ciencia de dato...","19,649","-101,225",NaN
165,Trabajo Social - ENTS,"Escuela Nacional de Trabajo Social, Ciudad de ...",Ciencias Sociales,La demanda a primer ingreso en esta licenciatu...,ENTS: 63,ENTS: 7.45,Asignación total de 578 lugares,Haber cursado el Área de las Ciencias Sociales...,El trabajador social es el profesional que inv...,"trabajo social, intervención social, bienestar...","19,3213","-99,1861",NaN
166,Traducción - ENES Morelia,"ENES Unidad Morelia, Michoacán",Humanidades y de las Artes,La demanda a primer ingreso en esta licenciatu...,ENES Unidad Morelia: 69,ENES Unidad Morelia: 8.00,Información no disponible,Haber cursado el Área de las Humanidades y Art...,El traductor es el profesional que transfiere ...,"traducción, traductor, interpretación, lenguas...","19,649","-101,225",NaN
167,Turismo y Desarrollo Sostenible - ENES Mérida,"ENES Unidad Mérida, Yucatán",Ciencias Sociales,La demanda a primer ingreso en esta licenciatu...,ENES Unidad Mérida: 60,ENES Unidad Mérida: 7.62,Asignación total de 69 lugares,Haber cursado el Área de las Ciencias Sociales...,El licenciado en Turismo y Desarrollo Sost

In [35]:
df.columns

Index(['Nombre de la carrera', 'Dirección', 'Área del conocimiento',
       'Datos Estadísticos (DGAE-UNAM)',
       'Aciertos Mínimos por Examen de Selección',
       'Promedio de corte pase reglamentado', 'Cupo',
       'Características del aspirante', 'Descripción de la carrera',
       'Palabras asociadas', 'Latitud', 'Longitud', 'Unnamed: 12'],
      dtype='str')

In [28]:
# 1 Obtener ubicación del usuario
def obtener_ubicacion():
    url = "http://ip-api.com/json/"
    response = requests.get(url)
    data = response.json()
    
    lat = data['lat']
    lon = data['lon']
    
    return (lat, lon)


In [29]:
# 2 Cargar datos del CSV

def cargar_datos(ruta):
    df = pd.read_csv(ruta)

    df['Latitud'] = (
        df['Latitud']
        .astype(str)
        .str.replace(',', '.')
        .str.strip()
    )

    df['Longitud'] = (
        df['Longitud']
        .astype(str)
        .str.replace(',', '.')
        .str.strip()
    )

    df['Latitud'] = pd.to_numeric(df['Latitud'], errors='coerce')
    df['Longitud'] = pd.to_numeric(df['Longitud'], errors='coerce')

    # Eliminar filas con datos inválidos
    df = df.dropna(subset=['Latitud', 'Longitud'])

    return df



In [32]:

#3 Calcular universidad más cercana

def universidad_mas_cercana(df, ubicacion_usuario,n=3):
    distancias = []

    for _, fila in df.iterrows():
        ubicacion_uni = (fila['Latitud'], fila['Longitud'])
        distancia = geodesic(ubicacion_usuario, ubicacion_uni).km
        distancias.append(distancia)

    df['Distancia_km'] = distancias

    # Ordenar por distancia
    df_ordenado = df.sort_values(by='Distancia_km')

    return df_ordenado.head(n)


In [37]:
#4. Ejecutar programa

def main():
    archivo = "carreras.csv"

    df = cargar_datos(archivo)
    mi_ubicacion = obtener_ubicacion()

    resultados = universidad_mas_cercana(df, mi_ubicacion,3)

    print("Tu ubicación:", mi_ubicacion)
    print("\nUniversidades más cercanas a ti:")

    for i, (_, fila) in enumerate(resultados.iterrows(), start=1):
        print("Carrera:", fila['Nombre de la carrera'])     
        print("Distancia:", round(fila['Distancia_km'], 2), "km\n")

# Ejecutar
if __name__ == "__main__":
    main()


Tu ubicación: (19.6815, -101.2036)

Universidades más cercanas a ti:
Carrera: Administración de Archivos y Gestión Documental - ENES Morelia (Distancia)
Distancia: 4.24 km

Carrera: Ciencias Agroforestales - ENES Morelia
Distancia: 4.24 km

Carrera: Ciencias Ambientales - ENES Morelia
Distancia: 4.24 km

